In [1]:
pip install --upgrade astrapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.4/353.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Suc

In [2]:
!pip install --quiet langchain langchain-openai langchain-community cassio datasets evaluate nltk rouge-score langsmith

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.13.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requ

In [3]:
from astrapy import DataAPIClient

# Initialize the client
client = DataAPIClient()
db = client.get_database(
  "https://e87070a9-10fc-406e-9f87-84346d940db7-us-east-2.apps.astra.datastax.com",
  token="AstraCS:hYvmiNGSbdbudDdIBZJSkYNh:f6445991617a37a71342d969f4907730049d1b045e3e52d04c212929040a0e50"
)

print(f"Connected to Astra DB: {db.list_collection_names()}")

Connected to Astra DB: []


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

# Ecosystem and LLM Keys
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_TRACING"] = "true"

# Astra DB Credentials
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT", "")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN", "")
ASTRA_DB_KEYSPACE = os.getenv("ASTRA_DB_KEYSPACE", "default_keyspace")

In [12]:
!pip install -U langchain-astradb

In [21]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_astradb import AstraDBVectorStore # Changed import
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load data from target documentation
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# 2. Text Splitting
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=25)
doc_splits = text_splitter.split_documents(docs_list)

# 3. Initialize Astra DB Vector Store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = AstraDBVectorStore( # Changed class name
    embedding=embeddings,
    collection_name="rag_evaluation_collection",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_KEYSPACE
)

# Insert splits safely into Astra DB
vectorstore.add_documents(doc_splits)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [25]:
retriever.invoke("what is agents")

[Document(id='bc74576cbb3c4934a5c696a12f45cca7', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, ther

In [28]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7dd992b7e450>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7ddb256eecc0>, root_client=<openai.OpenAI object at 0x7ddb256c95e0>, root_async_client=<openai.AsyncOpenAI object at 0x7dd97b83aba0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'

In [29]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}



In [30]:
rag_bot("What is agents")

{'answer': 'Agents refer to entities that can act and make decisions autonomously in an environment, often powered by advanced algorithms or models like Large Language Models (LLMs). In the context of the provided documents, generative agents simulate human behavior in a sandbox environment, allowing them to interact and exhibit social behaviors. They utilize memory and planning mechanisms to influence their interactions based on past experiences.',
 'documents': [Document(id='bc74576cbb3c4934a5c696a12f45cca7', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general prob

In [32]:
print("\n--- Manual Prompt Testing Active (Type 'exit' to stop) ---")
while True:
    user_prompt = input("Ask your RAG a question: ").strip()
    if user_prompt.lower() in ["exit", "quit"]:
        break
    if not user_prompt:
        continue

    res = rag_bot(user_prompt)
    print(f"\n[ANSWER]: {res['answer']}")
    print(f"[RETIREVED CHUNKS]: {res['documents']} documents fetched from Astra DB.\n")


--- Manual Prompt Testing Active (Type 'exit' to stop) ---
Ask your RAG a question: summerize agents

[ANSWER]: Agents are virtual entities controlled by language models (LLMs) that simulate human behavior and interactions in environments, such as the sandbox environment of the Generative Agents experiment. They utilize a combination of memory, planning, and reflection mechanisms to behave based on past experiences and to engage with other agents. Their capabilities can be evaluated through benchmarks that assess their tool use abilities across various levels, including calling APIs, retrieving information, and planning complex tasks.
[RETIREVED CHUNKS]: [Document(id='bc74576cbb3c4934a5c696a12f45cca7', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT

# EVALUATION

In [33]:
import evaluate
import torch
import pandas as pd
from typing_extensions import TypedDict, Annotated
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from langchain_openai import ChatOpenAI
from langsmith import Client

# Evaluators or Metrics
**Correctness: Response vs reference answer**

Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"

Mode: Requires a ground truth (reference) answer supplied through a dataset

Evaluator: Use LLM-as-judge to assess answer correctness.

In [34]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm=ChatOpenAI(model="gpt-4o-mini",temperature=0).with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]







### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [15]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Groundedness: Response vs retrieved docs

Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [16]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

## Retrieval Relevance: Retrieved docs vs input

In [17]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

BLEU score → Measures exact word/phrase overlap with the reference answer.
Higher = more similar wording
Lower = different wording

ROUGE-L score → Measures content/sequence similarity with the reference answer.
Higher = closer content structure
Lower = less overlap

Perplexity → Measures how confident/natural the model’s generated text is.
Lower = more fluent/confident
Higher = more uncertain/noisy

In [37]:
# =====================================================================
# 2. ADDITIONAL TRADITIONAL METRICS (BLEU, ROUGE, PERPLEXITY)
# =====================================================================
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

perplexity_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
perplexity_model = GPT2LMHeadModel.from_pretrained("gpt2")
if torch.cuda.is_available():
    perplexity_model = perplexity_model.cuda()

def calculate_perplexity(text: str) -> float:
    if not text.strip(): return 0.0
    encodings = perplexity_tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids
    if torch.cuda.is_available(): input_ids = input_ids.cuda()
    with torch.no_grad():
        outputs = perplexity_model(input_ids, labels=input_ids)
    return float(torch.exp(outputs.loss).item())

# Custom LangSmith wrapper for traditional text alignments
def traditional_string_metrics(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    student_ans = outputs.get("answer", "")
    ground_truth = reference_outputs.get("answer", "")

    try:
        b_score = float(bleu_metric.compute(predictions=[student_ans], references=[[ground_truth]]).get("bleu", 0.0))
    except: b_score = 0.0

    try:
        r_score = float(rouge_metric.compute(predictions=[student_ans], references=[ground_truth]).get("rougeL", 0.0))
    except: r_score = 0.0

    ppl_score = calculate_perplexity(student_ans)

    return {
        "results": [
            {"key": "bleu_score", "score": b_score},
            {"key": "rougeL_score", "score": r_score},
            {"key": "perplexity", "score": ppl_score}
        ]
    }

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
# =====================================================================
# 3. RUN THE COMBINED EVALUATION MATRIX
# =====================================================================
client = Client()
dataset_name = "Production RAG Evaluation Dataset"

# Define target data pairs
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

# Safeguard data generation block
try:
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
except Exception:
    pass

def target_runner(inputs: dict) -> dict:
    # Direct wrapper to target your exact rag_bot function
    return rag_bot(inputs["question"])

# Execute Evaluation Suite
experiment_results = client.evaluate(
    target_runner,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        retrieval_relevance,
        traditional_string_metrics
    ],
    experiment_prefix="comprehensive-rag-eval"
)

# Render summary metrics safely to a dataframe to prevent KeyErrors
df = experiment_results.to_pandas()

desired_columns = [
    "inputs.question", "outputs.answer",
    "feedback.correctness", "feedback.relevance", "feedback.groundedness", "feedback.retrieval_relevance",
    "feedback.bleu_score", "feedback.rougeL_score", "feedback.perplexity"
]
safe_columns = [col for col in desired_columns if col in df.columns]

pd.set_option('display.max_columns', None)
print(df[safe_columns])


View the evaluation results for experiment: 'comprehensive-rag-eval-e997594d' at:
https://smith.langchain.com/o/884be025-e7d4-4768-87a8-6ac71dc6e100/datasets/10ee6aff-f674-4c58-9f1e-6bc99ae29c7b/compare?selectedSessions=70d5ab71-8be4-4b5a-b6fd-4d981a0e57ab




0it [00:00, ?it/s]

                                     inputs.question  \
0        What are five types of adversarial attacks?   
1  What are the types of biases that can arise wi...   
2     How does the ReAct agent use self-reflection?    

                                      outputs.answer  feedback.correctness  \
0  The five types of adversarial attacks are toke...                  True   
1  The source documents provided do not specify t...                 False   
2  The ReAct agent uses self-reflection by incorp...                  True   

   feedback.relevance  feedback.groundedness  feedback.retrieval_relevance  \
0                True                   True                          True   
1               False                   True                         False   
2                True                   True                          True   

   feedback.bleu_score  feedback.rougeL_score  feedback.perplexity  
0             0.135182               0.844444            96.101448  
1          